# Companion Notebook 11: End-to-End LLM Generation Loop

This notebook implements a complete, self-contained **End-to-End LLM Generation Loop** in PyTorch, mapping strings to tokens, executing attention with a KV Cache, scaling logits, and sampling tokens autoregressively.


## 1. Pipeline Implementation
We implement all components from scratch:
- Tokenizer mapping
- Embedding layer
- Single-layer Self-Attention with a KV Cache
- Gating and Sampling (Temperature, Top-k, Repetition Penalty)
- Termination check (EOS token)


In [1]:
import torch
import torch.nn as nn
import numpy as np
import random

# Set random seed
torch.manual_seed(42)
np.random.seed(42)
random.seed(42)

# 1. Custom Vocabulary and Tokenizer
vocab = ["the", "cat", "sat", "on", "mat", "is", "soft", "mammal", "on_mat", "</s>"]
word_to_id = {w: i for i, w in enumerate(vocab)}
id_to_word = {i: w for i, w in enumerate(vocab)}
vocab_size = len(vocab)
EOS_ID = word_to_id["</s>"]

print("Vocabulary size:", vocab_size)
print("EOS Token ID:", EOS_ID)


Vocabulary size: 10
EOS Token ID: 9


### Explanation of Outputs (Vocabulary)
- **Vocabulary size**: 10 tokens are present.
- **EOS ID**: 9 represents the end-of-sentence marker (`</s>`).


In [2]:
# 2. Embedding Layer and Positional Additions
embed_dim = 8
max_len = 100

token_embeddings = nn.Embedding(vocab_size, embed_dim)
pos_embeddings = nn.Embedding(max_len, embed_dim)

# Initialize weights to match hand calculation values approximately
with torch.no_grad():
    token_embeddings.weight.data.uniform_(-0.5, 0.5)
    pos_embeddings.weight.data.uniform_(-0.1, 0.1)

# Define simple projection head (LM head)
lm_head = nn.Linear(embed_dim, vocab_size, bias=False)


In [3]:
# 3. Simple Single-Layer Attention with KV Cache
class KVAttentionLayer(nn.Module):
    def __init__(self, dim):
        super(KVAttentionLayer, self).__init__()
        self.dim = dim
        self.w_q = nn.Linear(dim, dim, bias=False)
        self.w_k = nn.Linear(dim, dim, bias=False)
        self.w_v = nn.Linear(dim, dim, bias=False)
        
    def forward(self, x_last, K_cache=None, V_cache=None):
        # x_last shape: [1, 1, dim] (the latest generated token)
        q = self.w_q(x_last) # [1, 1, dim]
        k = self.w_k(x_last) # [1, 1, dim]
        v = self.w_v(x_last) # [1, 1, dim]
        
        # Append to caches if they exist
        if K_cache is not None:
            K_cache = torch.cat([K_cache, k], dim=1)
            V_cache = torch.cat([V_cache, v], dim=1)
        else:
            K_cache = k
            V_cache = v
            
        # Compute scaled attention scores
        # q shape: [1, 1, dim]
        # K_cache shape: [1, seq_len, dim]
        scores = torch.matmul(q, K_cache.transpose(-2, -1)) / np.sqrt(self.dim) # [1, 1, seq_len]
        weights = torch.softmax(scores, dim=-1)
        
        # Weighted sum: out shape [1, 1, dim]
        out = torch.matmul(weights, V_cache)
        return out, K_cache, V_cache

attention_block = KVAttentionLayer(dim=embed_dim)


### Explanation of Outputs (Attention Block)
- **KVAttentionLayer**: Linear projections project inputs to query, key, and value vectors of dimension 8. Keys and values are concatenated sequentially with the historical cache.


In [4]:
# 4. Gating & Sampling Function
def sample_next_token(logits, generated_tokens, temp=0.7, top_k=3, penalty=1.2):
    # Detach and clone to prevent modifying grad graphs
    logits = logits.detach().clone().squeeze() # Ensure it is strictly 1D [vocab_size]
    
    # Apply repetition penalty to logits of already generated tokens
    for token_id in generated_tokens:
        val = logits[token_id].item()
        if val > 0:
            logits[token_id] = val / penalty
        else:
            logits[token_id] = val * penalty
            
    # Scale by temperature
    logits = logits / temp
    
    # Top-k pruning
    top_logits, top_indices = torch.topk(logits, top_k)
    pruned_logits = torch.full_like(logits, float('-inf'))
    pruned_logits[top_indices] = top_logits
    
    # Softmax probabilities
    probs = torch.softmax(pruned_logits, dim=-1)
    
    # Sample token ID
    sampled_id = torch.multinomial(probs, num_samples=1).item()
    return sampled_id, probs.numpy()


### Explanation of Outputs (Gating & Sampling)
- **sample_next_token**: Gating coefficients are scaled using the repetition penalty divisor, temperature-divided to shape entropy, masked via top-k, and evaluated using PyTorch multinomial sampling to generate the token ID.


In [5]:
# 5. Full Generation Loop execution
prompt_words = ["the", "cat", "sat"]
prompt_ids = [word_to_id[w] for w in prompt_words]
print("Prompt tokens:", prompt_words, "-> IDs:", prompt_ids)

# List to store all generated token IDs
gen_ids = list(prompt_ids)
generated_token_set = set(prompt_ids)

# Prefill Phase
K_cache, V_cache = None, None
for idx, p_id in enumerate(prompt_ids):
    # Embed token and position using standard 2D indexes
    x_emb = token_embeddings(torch.tensor([[p_id]])) # [1, 1, dim]
    p_emb = pos_embeddings(torch.tensor([[idx]]))    # [1, 1, dim]
    z_in = x_emb + p_emb
    # Forward attention pass (populating cache)
    out, K_cache, V_cache = attention_block(z_in, K_cache, V_cache)

print("\nKV Cache initialized during prefill:")
print("  Key Cache Shape:  ", list(K_cache.shape))
print("  Value Cache Shape:", list(V_cache.shape))


Prompt tokens: ['the', 'cat', 'sat'] -> IDs: [0, 1, 2]

KV Cache initialized during prefill:
  Key Cache Shape:   [1, 3, 8]
  Value Cache Shape: [1, 3, 8]


### Explanation of Outputs (Prefill Phase)
- **KV Cache shapes**: `[1, 3, 8]` represents batch size 1, sequence length 3, and hidden dimension 8. All prompt tokens are processed in parallel, populating the initial caches.


In [6]:
# Decode Phase
max_gen_steps = 5

for step in range(max_gen_steps):
    # Embedding of the latest token in the sequence using standard 2D indexes
    latest_id = gen_ids[-1]
    curr_pos = len(gen_ids) - 1
    
    x_emb = token_embeddings(torch.tensor([[latest_id]])) # [1, 1, dim]
    p_emb = pos_embeddings(torch.tensor([[curr_pos]]))    # [1, 1, dim]
    z_in = x_emb + p_emb
    
    # Attention block with KV Cache updates
    out, K_cache, V_cache = attention_block(z_in, K_cache, V_cache)
    
    # Logits head projection
    logits = lm_head(out).squeeze() # Should result in shape [vocab_size]
    
    # Sample next token
    next_id, probs_dist = sample_next_token(logits, generated_token_set, temp=0.8, top_k=2)
    
    # Append
    gen_ids.append(next_id)
    generated_token_set.add(next_id)
    
    print(f"\nStep {step+1:02d} | Generated Token: {id_to_word[next_id]!r:8s} | ID: {next_id:2d}")
    print("Probabilities distribution:", np.round(probs_dist, 4))
    
    # Check termination
    if next_id == EOS_ID:
        print("EOS token generated. Stopping...")
        break

final_sequence = [id_to_word[t_id] for t_id in gen_ids]
print("\n=== Final Generated Sequence ===")
print(" ".join(final_sequence))



Step 01 | Generated Token: 'mat'    | ID:  4
Probabilities distribution: [0.4616 0.     0.     0.     0.5384 0.     0.     0.     0.     0.    ]

Step 02 | Generated Token: 'the'    | ID:  0
Probabilities distribution: [0.4829 0.     0.     0.     0.5171 0.     0.     0.     0.     0.    ]

Step 03 | Generated Token: 'the'    | ID:  0
Probabilities distribution: [0.4824 0.     0.     0.     0.5176 0.     0.     0.     0.     0.    ]

Step 04 | Generated Token: 'the'    | ID:  0
Probabilities distribution: [0.4826 0.     0.     0.     0.5174 0.     0.     0.     0.     0.    ]

Step 05 | Generated Token: 'the'    | ID:  0
Probabilities distribution: [0.4827 0.     0.     0.     0.5173 0.     0.     0.     0.     0.    ]

=== Final Generated Sequence ===
the cat sat mat the the the the


### Explanation of Outputs (Decode Phase)
- **Step logs**: Tokens are generated one-by-one autoregressively. Each step appends a new token to the sequence and updates the KV Cache. Gating probabilities shift dynamically to prevent repetition loops. The final sequence prints `the cat sat mat the the the the` indicating autoregressive looping.
